# A simple machine learning pipeline

A small end-to-end example with a real branch point: one dataset, one
cleaning step, then three model variants trained side by side — and the store
picks the winner. Standard library only, so the "model" is a nearest-centroid
classifier on synthetic 2-D data; the shape of the workflow is the point.

In [1]:
import json
import math
import random
import shutil
import tempfile
from pathlib import Path

from ancestree import LineageStore

workdir = Path(tempfile.mkdtemp(prefix="ancestree-ml-"))
store = LineageStore(
    workdir / "ml",
    rules={"clean": ["ingest"], "train": ["clean"], "evaluate": ["train"]},
)

def write_points(path, points):
    path.write_text("\n".join(f"{x:.4f},{y:.4f},{label}" for x, y, label in points))

def read_points(path):
    rows = []
    for line in path.read_text().splitlines():
        x, y, label = line.split(",")
        rows.append((float(x), float(y), int(label)))
    return rows

In [2]:
# ingest: two noisy clusters, with a few corrupt rows thrown in
rng = random.Random(7)
points = []
for label, (cx, cy) in enumerate([(0.0, 0.0), (3.0, 3.0)]):
    for _ in range(200):
        points.append((rng.gauss(cx, 1.0), rng.gauss(cy, 1.0), label))
for _ in range(12):  # corruption: impossible coordinates
    points.append((rng.uniform(50, 60), rng.uniform(50, 60), rng.randint(0, 1)))
rng.shuffle(points)

with store.create_node(step_type="ingest") as ingest:
    write_points(ingest / "points.csv", points)
    ingest.add_meta("rows", len(points))

with store.create_node(step_type="clean", parent=ingest) as clean:
    [raw] = store.from_parent(clean, "points.csv")
    good = [p for p in read_points(raw) if abs(p[0]) < 10 and abs(p[1]) < 10]
    write_points(clean / "points.csv", good)
    clean.add_meta("rows", len(good))
    clean.add_meta("dropped", len(points) - len(good), group="Quality")

print(f"ingested {len(points)} rows, kept {len(good)} after cleaning")

ingested 412 rows, kept 400 after cleaning


## The branch point

Three `train` nodes hang off the same `clean` node, one per hyperparameter.
Each records its accuracy as searchable metadata, so the comparison is a
query rather than a spreadsheet.

In [3]:
def train_and_score(rows, trim):
    # nearest-centroid, with an optional trim of the furthest points
    split = int(len(rows) * 0.8)
    train, test = rows[:split], rows[split:]
    centroids = {}
    for label in (0, 1):
        cluster = [(x, y) for x, y, lab in train if lab == label]
        cx = sum(x for x, _ in cluster) / len(cluster)
        cy = sum(y for _, y in cluster) / len(cluster)
        if trim:  # drop the furthest fraction and re-fit
            cluster.sort(key=lambda p: (p[0] - cx) ** 2 + (p[1] - cy) ** 2)
            cluster = cluster[: int(len(cluster) * (1 - trim))]
            cx = sum(x for x, _ in cluster) / len(cluster)
            cy = sum(y for _, y in cluster) / len(cluster)
        centroids[label] = (cx, cy)
    correct = 0
    for x, y, label in test:
        predicted = min(centroids, key=lambda lab: math.dist((x, y), centroids[lab]))
        correct += predicted == label
    return centroids, correct / len(test)

trained = []
for trim in (0.0, 0.1, 0.3):
    with store.create_node(step_type="train", parent=clean) as node:
        centroids, accuracy = train_and_score(good, trim)
        (node / "model.json").write_text(json.dumps(centroids))
        node.add_meta("trim", trim, group="Params")
        node.add_meta("accuracy", round(accuracy, 4), group="Metrics")
        trained.append(node)

for node in trained:
    record = store.get(node)
    print(f"trim={record.metadata['trim']['value']:<4} accuracy={record.metadata['accuracy']['value']}")

trim=0.0  accuracy=0.975
trim=0.1  accuracy=0.975
trim=0.3  accuracy=0.975


## Picking the winner is a query

In [4]:
candidates = store.find(step_type="train", accuracy=lambda a: a is not None)
best = max(candidates, key=lambda n: n.metadata["accuracy"]["value"])
print("best:", best.node_id, "accuracy", best.metadata["accuracy"]["value"])

with store.create_node(step_type="evaluate", parent=best) as final:
    [model_file] = store.from_parent(final, "model.json")
    final.add_meta("chosen_model", best.node_id)
    final.add_meta("verdict", "ship it")

print("evaluation lineage:", " -> ".join(n.step_type for n in store.lineage(final)))
print("which cleaning produced it:", [n.node_id for n in store.ancestors(final, step_type="clean")])

best: 7d75c2b6 accuracy 0.975
evaluation lineage: ingest -> clean -> train -> evaluate
which cleaning produced it: ['7ba6306e']


## The run comparison, straight from SQL

In [5]:
for row in store.sql(
    "SELECT n.node_id, m.num_value AS accuracy FROM node n "
    "JOIN metadata m ON m.node_id = n.node_id AND m.key = 'accuracy' "
    "WHERE n.step_type = 'train' ORDER BY m.num_value DESC"
):
    print(f"{row['node_id']}  accuracy={row['accuracy']}")

store.close()
shutil.rmtree(workdir)
print("cleaned up")

7d75c2b6  accuracy=0.975
06b20912  accuracy=0.975
8e73012c  accuracy=0.975
cleaned up
